# Amazon Review Helpfulness Prediction
This notebook walks through the full pipeline for predicting whether an Amazon Electronics review will be considered helpful by other customers.

**Models:** XGBoost (classical baseline) and DistilBERT (transformer)

**Target:** Binary classification — helpful (3+ votes) vs. not helpful

**Note on label definition:** the original proposal used a helpful_vote / total_vote ratio thresholded at 0.7. The dataset actually used (Amazon Reviews 2023) has no total_vote field, only a raw helpful_vote count — so the label here is a direct threshold on helpful_vote instead. See METHODS.docx in the repo for full justification.

## 1. Setup
Install dependencies if not already installed in your environment.

In [ ]:
# Run once per environment. Safe to skip if already installed.
!pip install xgboost textstat vaderSentiment --break-system-packages

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import textstat
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel

print('All imports successful')

## 2. Data Pipeline
Load the `Electronics.jsonl` file, filter records, create binary labels, and split into train/val/test sets.

- **Source:** Amazon Reviews 2023, Electronics category (~20M raw records)
- **Sampling:** 200,000 records streamed from the raw file (avoids loading the full ~20M into memory)
- **Label:** `helpful = 1` if `helpful_vote >= 3`, else `0`
- **Split:** 70% train, 15% val, 15% test (stratified to preserve class balance)

In [ ]:
DATA_PATH = "/Users/shail2169/Electronics.jsonl"  # update path if needed
SAMPLE_SIZE = 200_000        # records to sample from the raw ~20M
HELPFULNESS_THRESHOLD = 3    # min helpful_vote count to count as "helpful"
RANDOM_STATE = 42            # fixed seed for reproducibility

records = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        # skip records missing text or title — can't extract features from them
        if not record.get("text") or not record.get("title"):
            continue
        records.append({
            "text": record["text"],
            "review_title": record["title"],
            "rating": record["rating"],
            "helpful_vote": record["helpful_vote"],
            "verified_purchase": record["verified_purchase"],
        })
        if len(records) >= SAMPLE_SIZE:
            break

df = pd.DataFrame(records)
df["helpful"] = (df["helpful_vote"] >= HELPFULNESS_THRESHOLD).astype(int)
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['helpful'].value_counts()}")
print(f"Positive rate: {df['helpful'].mean():.2%}")

In [ ]:
# Stratified 70/15/15 split — keeps the same helpful/not-helpful ratio in every split
train, temp = train_test_split(df, test_size=0.30, random_state=RANDOM_STATE, stratify=df["helpful"])
val, test = train_test_split(temp, test_size=0.50, random_state=RANDOM_STATE, stratify=temp["helpful"])
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

## 3. Feature Engineering
Extract NLP-derived features from review text and combine with metadata features.

- **Text structure:** length, word count, lexical diversity, avg word length
- **Readability:** Flesch Reading Ease, Flesch-Kincaid Grade (higher = easier to read)
- **Sentiment:** VADER scores (positive, negative, neutral, compound)
- **Metadata:** star rating, verified purchase, title length

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def extract_features(df):
    """Compute NLP + metadata features for a DataFrame of reviews."""
    df = df.copy()
    df = df.dropna(subset=["text", "review_title"]).reset_index(drop=True)

    # text structure
    df["review_length"] = df["text"].str.len()
    df["word_count"] = df["text"].apply(lambda x: len(x.split()))
    df["avg_word_length"] = df["text"].apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
    df["title_length"] = df["review_title"].str.len()

    # lexical diversity = unique words / total words
    df["lexical_diversity"] = df["text"].apply(lambda x: len(set(x.lower().split())) / len(x.split()) if x.split() else 0)

    # readability
    df["flesch_reading_ease"] = df["text"].apply(textstat.flesch_reading_ease)
    df["flesch_kincaid_grade"] = df["text"].apply(textstat.flesch_kincaid_grade)

    # VADER sentiment
    df["vader_positive"] = df["text"].apply(lambda x: analyzer.polarity_scores(x)["pos"])
    df["vader_negative"] = df["text"].apply(lambda x: analyzer.polarity_scores(x)["neg"])
    df["vader_neutral"] = df["text"].apply(lambda x: analyzer.polarity_scores(x)["neu"])
    df["vader_compound"] = df["text"].apply(lambda x: analyzer.polarity_scores(x)["compound"])

    # metadata
    df["verified_purchase"] = df["verified_purchase"].astype(int)
    df["rating"] = df["rating"].astype(float)
    return df

print("Extracting features for train split (this may take a few minutes)...")
train_feat = extract_features(train)
print("Extracting features for val split...")
val_feat = extract_features(val)
print("Extracting features for test split...")
test_feat = extract_features(test)
print("Done")

## 4. Classical Model — XGBoost
Train an XGBoost classifier on the engineered features. Sample weighting (instead of oversampling) is used to handle the class imbalance without introducing duplicated feature rows.

In [ ]:
FEATURE_COLS = [
    "review_length", "word_count", "avg_word_length", "title_length",
    "lexical_diversity", "flesch_reading_ease", "flesch_kincaid_grade",
    "vader_positive", "vader_negative", "vader_neutral", "vader_compound",
    "verified_purchase", "rating"
]

X_train, y_train = train_feat[FEATURE_COLS], train_feat["helpful"]
X_val, y_val = val_feat[FEATURE_COLS], val_feat["helpful"]
X_test, y_test = test_feat[FEATURE_COLS], test_feat["helpful"]

# balanced sample weights compensate for the imbalanced positive class
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=50
)

In [ ]:
# Evaluate XGBoost — F1 macro is used instead of accuracy since accuracy
# would look artificially high on this imbalanced dataset.
for name, X, y in [("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    preds = xgb_model.predict(X)
    probs = xgb_model.predict_proba(X)[:, 1]
    print(f"\n── {name} ──")
    print(f"F1 Macro: {f1_score(y, preds, average='macro'):.4f}")
    print(f"AUC-ROC:  {roc_auc_score(y, probs):.4f}")
    print(classification_report(y, preds, target_names=["not helpful", "helpful"]))

In [ ]:
# Feature importance shows which signals XGBoost relies on most —
# useful for interpretability, which the transformer model lacks.
importance = dict(zip(FEATURE_COLS, xgb_model.feature_importances_))
sorted_importance = dict(sorted(importance.items(), key=lambda x: x[1], reverse=True))
print("── Feature Importance ──")
for feat, score in sorted_importance.items():
    print(f"{feat:<30} {score:.4f}")

## 5. Transformer Model — DistilBERT
Fine-tune a DistilBERT classifier with **frozen transformer weights** — only the classification head is trained. This is a deliberate tradeoff to make training feasible on CPU-only hardware.

Due to CPU constraints, we train on a stratified subsample of 10,000 records rather than the full 140k training set.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-4
TRAIN_SAMPLE = 10_000
VAL_SAMPLE = 3_000

# Stratified subsample — keeps a 50/50 class balance in the smaller training set
# (the full dataset is ~92/8, which would make CPU training on 10k records
# yield almost no positive examples per batch otherwise)
train_pos = train_feat[train_feat["helpful"] == 1].sample(min(TRAIN_SAMPLE // 2, train_feat["helpful"].sum()), random_state=RANDOM_STATE)
train_neg = train_feat[train_feat["helpful"] == 0].sample(TRAIN_SAMPLE // 2, random_state=RANDOM_STATE)
train_sub = pd.concat([train_pos, train_neg]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

val_pos = val_feat[val_feat["helpful"] == 1].sample(min(VAL_SAMPLE // 2, val_feat["helpful"].sum()), random_state=RANDOM_STATE)
val_neg = val_feat[val_feat["helpful"] == 0].sample(VAL_SAMPLE // 2, random_state=RANDOM_STATE)
val_sub = pd.concat([val_pos, val_neg]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Train subsample: {len(train_sub)}, Val subsample: {len(val_sub)}")
print(f"Train class distribution:\n{train_sub['helpful'].value_counts()}")

In [ ]:
class ReviewDataset(Dataset):
    """Wraps review texts + labels into tokenized tensors for DistilBERT."""
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

class DistilBertClassifier(nn.Module):
    """DistilBERT encoder (frozen) + a small trainable classification head."""
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(MODEL_NAME)
        # freeze all transformer weights — only the head below will train
        for param in self.bert.parameters():
            param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # use the [CLS] token's hidden state as the pooled sentence representation
        cls_output = output.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

print("Classes defined")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
train_dataset = ReviewDataset(train_sub["text"].tolist(), train_sub["helpful"].tolist(), tokenizer)
val_dataset = ReviewDataset(val_sub["text"].tolist(), val_sub["helpful"].tolist(), tokenizer)
test_dataset = ReviewDataset(test_feat["text"].tolist(), test_feat["helpful"].tolist(), tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model = DistilBertClassifier().to(device)

# class weights fed into the loss function to further counter imbalance
class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_sub["helpful"].values)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# only the unfrozen classification head parameters get optimized
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

print("Model and data loaders ready")

In [ ]:
# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# Evaluate transformer — same metrics as XGBoost for direct comparison
def evaluate_transformer(model, loader, device, split_name):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"]
            outputs = model(input_ids, attention_mask)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
    print(f"\n── {split_name} ──")
    print(f"F1 Macro: {f1_score(all_labels, all_preds, average='macro'):.4f}")
    print(f"AUC-ROC:  {roc_auc_score(all_labels, all_probs):.4f}")
    print(classification_report(all_labels, all_preds, target_names=["not helpful", "helpful"]))

evaluate_transformer(model, val_loader, device, "Validation")
evaluate_transformer(model, test_loader, device, "Test")